# DATA-549: In-Class Assignment 11

# Exercise 1 (10 points)

The following question is based on Section 8.3 (Self-Attention and the Transformer Block).

Implement **scaled dot-product self-attention from scratch using NumPy** (do not use `spacy`, `transformers`, or any deep learning library for this exercise).

You are given four token embeddings that have already been projected into query/key space ($d_k = 3$) and value space ($d_v = 2$):

```python
import numpy as np

Q = K = np.array([[1., 0., 1.],
                   [0., 1., 1.],
                   [1., 1., 0.],
                   [0., 0., 1.]])

V = np.array([[8., 0.],
              [0., 8.],
              [4., 4.],
              [2., 2.]])
```

Write code that:

1. Defines a `softmax(x, axis=-1)` function that is numerically stable (subtracts the row max before exponentiating).
2. Defines a `self_attention(Q, K, V)` function that implements

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

   and returns both the output matrix and the attention weight matrix.
3. Applies `self_attention` to `Q`, `K`, `V` above, and prints the attention weight matrix (rounded to 3 decimals) and the output matrix (rounded to 3 decimals).
4. Verifies, with an assertion or printed check, that every row of the attention weight matrix sums to 1.
5. In a short comment or print statement, state which token(s) token 1 (row 0) attends to the most, and explain *why* in one sentence by relating it to the dot product between its query and the corresponding keys.

In [1]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

Q = K = np.array([[1., 0., 1.],
                   [0., 1., 1.],
                   [1., 1., 0.],
                   [0., 0., 1.]])

V = np.array([[8., 0.],
              [0., 8.],
              [4., 4.],
              [2., 2.]])

# Compute self-attention output and weights
output, weights = self_attention(Q, K, V)

print("Attention Weights:")
print(np.round(weights, 3))

print("Attention Output:")
print(np.round(output, 3))

# Verify that attention weights sum to 1 along the last axis
print("Sum of attention weights along last axis:")
row_sums = weights.sum(axis=1)

print(np.round(row_sums, 3))

assert np.allclose(row_sums, 1.0)
print("Check passed: Attention weights sum to 1 along the last axis.")

Attention Weights:
[[0.373 0.209 0.209 0.209]
 [0.209 0.373 0.209 0.209]
 [0.23  0.23  0.41  0.129]
 [0.281 0.281 0.158 0.281]]
Attention Output:
[[4.235 2.928]
 [2.928 4.235]
 [3.741 3.741]
 [3.438 3.438]]
Sum of attention weights along last axis:
[1. 1. 1. 1.]
Check passed: Attention weights sum to 1 along the last axis.


# Exercise 2 (10 points)

The following question is based on Section 9.3.3 (TextRank).

**TextRank** turns extractive summarization into a graph-centrality problem: build a complete weighted graph in which each node is a sentence and each edge weight is the cosine similarity between two sentences, then run PageRank over that graph so that a sentence which is similar to many other important sentences receives a high score.

You are given the following helper function and source text:

```python
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def split_sentences(text):
    """Splits text into sentences without requiring an nltk download."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

text = (
    "Grand View University offers a Master of Science in Data Analytics. "
    "The program covers statistics, machine learning, and natural language processing. "
    "Students complete a capstone project applying these skills to a real dataset. "
    "Many graduates move into data science and analytics roles after finishing the program. "
    "The university also offers strong support services for working professionals. "
    "Class sizes are small, which allows close interaction between students and faculty."
)
```

Write a function `textrank_summary(text, k=3)` that:

1. Splits `text` into sentences using `split_sentences`.
2. Represents each sentence as a TF–IDF vector with `TfidfVectorizer(stop_words="english")`.
3. Computes the pairwise cosine similarity matrix between sentences, and sets the diagonal to `0.0` (a sentence must not vote for itself).
4. Builds a graph from the similarity matrix with `nx.from_numpy_array` and scores each sentence with `nx.pagerank(graph, alpha=0.85)`.
5. Selects the top-`k` highest-scoring sentences, **restores their original reading order**, and returns them joined into a single string.

Then call `textrank_summary(text, k=3)` on the `text` above and print the result.

In [2]:
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def split_sentences(text):
    """Splits text into sentences without requiring an nltk download."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

text = (
    "Grand View University offers a Master of Science in Data Analytics. "
    "The program covers statistics, machine learning, and natural language processing. "
    "Students complete a capstone project applying these skills to a real dataset. "
    "Many graduates move into data science and analytics roles after finishing the program. "
    "The university also offers strong support services for working professionals. "
    "Class sizes are small, which allows close interaction between students and faculty."
)

def textrank_summary(text, k=3):
    sents = split_sentences(text)
    if len(sents) <= k:
        return sents
    
    # Compute tf-idf matrix
    X = TfidfVectorizer(stop_words='english').fit_transform(sents)
    
    # Compute cosine similarity matrix
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)  # Remove self-similarity
    
    # Build graph and compute PageRank scores
    graph = nx.from_numpy_array(sim)
    scores = nx.pagerank(graph, alpha=0.85)
    
    # Select top-k sentences
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    
    return " ".join([sents[i] for i in sorted(top)])
    
# Example usage
print("Original text:")
print(text)

summary = textrank_summary(text, k=3)
print("\nSummary:")
print(summary)

Original text:
Grand View University offers a Master of Science in Data Analytics. The program covers statistics, machine learning, and natural language processing. Students complete a capstone project applying these skills to a real dataset. Many graduates move into data science and analytics roles after finishing the program. The university also offers strong support services for working professionals. Class sizes are small, which allows close interaction between students and faculty.

Summary:
Grand View University offers a Master of Science in Data Analytics. Students complete a capstone project applying these skills to a real dataset. Many graduates move into data science and analytics roles after finishing the program.
